# Summary_Day3.ipynb  
## PyTorch 기본 기능 · Tensor 핵심 · 자동 미분

이번 3강은 PyTorch에서 가장 기본이 되는 **Tensor**를 정리하는 강의다.

2강에서 NumPy 배열을 배웠다면, 3강은 그 배열 사고를 PyTorch Tensor로 확장하는 단계다.  
강의 자료에서도 PyTorch Tensor를 단순한 숫자 배열이 아니라, **자동 미분과 GPU 가속을 지원하는 딥러닝용 데이터 컨테이너**로 설명한다.

오늘 전체 흐름은 이렇게 보면 된다.

```text
NumPy ndarray
→ PyTorch Tensor
→ requires_grad
→ 계산 그래프
→ backward()
→ grad 확인
→ zero_() 초기화
```

이번 강의에서 꼭 잡아야 하는 포인트는 다음과 같다.

1. Tensor의 차원별 의미를 이해한다.
2. Tensor의 `shape`, `dtype`, `device`, `requires_grad`를 확인한다.
3. `view()`, `reshape()`, `squeeze()`, `unsqueeze()`로 Tensor 모양을 바꾼다.
4. `torch.max()`로 최댓값과 인덱스를 구한다.
5. NumPy 배열과 Tensor를 서로 변환한다.
6. `requires_grad=True`로 자동 미분을 준비한다.
7. `backward()`로 gradient를 계산한다.
8. gradient는 누적되므로 `zero_()`로 초기화해야 한다는 점을 이해한다.
9. 4차원 이미지 Tensor 구조 `(batch, channel, height, width)`를 이해한다.

> 필기 포인트:  
> Tensor는 NumPy array와 비슷해 보이지만, 딥러닝에서는 gradient 추적과 GPU 이동이 붙는다는 점이 핵심이다.

## 1. 라이브러리 준비

이번 실습에서는 NumPy, Matplotlib, PyTorch를 사용한다.

### 함수/모듈 사용법

```python
import numpy as np
import matplotlib.pyplot as plt
import torch
```

- `np`: NumPy 배열을 만들고 다룰 때 사용한다.
- `plt`: 그래프를 그릴 때 사용한다.
- `torch`: PyTorch Tensor와 자동 미분을 사용할 때 필요하다.

### 자주 쓰는 설정

```python
torch.manual_seed(123)
np.random.seed(123)
```

- 랜덤 결과를 고정한다.
- 같은 코드를 다시 실행해도 같은 난수가 나오게 만든다.

> 실습 메모:  
> 강의 원본에는 Colab 폰트 설치와 torchviz 설치 코드가 들어 있다.  
> 여기서는 모든 환경에서 바로 실행되도록 외부 설치 코드는 제외하고 핵심 Tensor 코드만 정리한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

%matplotlib inline

np.set_printoptions(suppress=True, precision=4)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

torch.manual_seed(123)
np.random.seed(123)

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)

## 2. Tensor란 무엇인가

Tensor는 PyTorch에서 데이터를 담는 기본 컨테이너다.

NumPy의 `ndarray`와 비슷하지만, PyTorch Tensor는 딥러닝에 필요한 기능을 더 가지고 있다.

```text
Tensor = 숫자 배열 + GPU 연산 + 자동 미분
```

강의에서 중요한 흐름은 이것이다.

```text
Raw Data
→ 숫자 배열
→ Tensor
→ 모델 연산
→ Loss
→ Gradient
```

> 기억할 점:  
> 딥러닝 모델은 결국 숫자로 된 데이터를 계속 변환하는 함수다.  
> 그래서 이 숫자 컨테이너인 Tensor를 먼저 이해해야 한다.

In [ ]:
concepts = {
    "Tensor": "PyTorch의 기본 숫자 컨테이너",
    "ndarray": "NumPy의 다차원 배열",
    "Autograd": "PyTorch 자동 미분 기능",
    "GPU": "대량 연산을 빠르게 처리하는 장치"
}

for key, value in concepts.items():
    print(f"{key}: {value}")

## 3. Tensor 차원 개념

Tensor는 차원 수에 따라 부르는 이름이 달라진다.

| 차원 | 이름 | 예시 |
|---|---|---|
| 0차원 | Scalar | `5` |
| 1차원 | Vector | `[1, 2, 3]` |
| 2차원 | Matrix | `[[1, 2], [3, 4]]` |
| 3차원 이상 | High-dimensional Tensor | 이미지, 비디오, 배치 데이터 |

> 헷갈림 포인트:  
> Tensor의 차원은 괄호나 대괄호가 몇 겹이냐와도 연결된다.  
> 하지만 실제로는 항상 `.shape`과 `.dim()`으로 확인하는 습관이 더 안전하다.

In [ ]:
scalar_np = np.array(10)
vector_np = np.array([1, 2, 3, 4, 5])
matrix_np = np.array([[1, 2, 3], [4, 5, 6]])
tensor_3d_np = np.arange(24).reshape(2, 3, 4)

print("scalar ndim:", scalar_np.ndim, "shape:", scalar_np.shape)
print("vector ndim:", vector_np.ndim, "shape:", vector_np.shape)
print("matrix ndim:", matrix_np.ndim, "shape:", matrix_np.shape)
print("tensor_3d ndim:", tensor_3d_np.ndim, "shape:", tensor_3d_np.shape)

## 4. 0차원 Tensor: Scalar

0차원 Tensor는 숫자 하나만 담고 있는 Tensor다.

### 함수 사용법: `torch.tensor()`

```python
torch.tensor(1.0)
```

- 괄호 안에 숫자, 리스트, NumPy 배열 등을 넣을 수 있다.
- 반환값은 PyTorch Tensor다.

### 함수 사용법: `.float()`

```python
tensor.float()
```

- Tensor의 데이터 타입을 `torch.float32`로 바꾼다.
- PyTorch Layer와 연산할 때 float 타입을 자주 사용한다.

In [ ]:
r0 = torch.tensor(1.0).float()

print("r0:", r0)
print("type:", type(r0))
print("dtype:", r0.dtype)
print("shape:", r0.shape)
print("dim:", r0.dim())
print("data:", r0.data)

출력 해석:

- `shape = torch.Size([])`이면 0차원 Tensor다.
- 차원이 없기 때문에 shape 안이 비어 있다.
- `dtype = torch.float32`는 32비트 실수형이라는 뜻이다.

> 필기 포인트:  
> Loss 값처럼 숫자 하나로 표현되는 값은 보통 0차원 Tensor로 나온다.

## 5. 1차원 Tensor: Vector

1차원 Tensor는 숫자가 한 줄로 나열된 구조다.

예:

```text
[1, 2, 3, 4, 5]
```

NumPy 배열을 Tensor로 바꾸는 흐름을 확인한다.

In [ ]:
r1_np = np.array([1, 2, 3, 4, 5])

r1 = torch.tensor(r1_np).float()

print("NumPy array:", r1_np)
print("NumPy shape:", r1_np.shape)

print("\nTensor:", r1)
print("dtype:", r1.dtype)
print("shape:", r1.shape)
print("dim:", r1.dim())

### 함수 사용법: `torch.tensor(np_array)`

```python
torch.tensor(r1_np)
```

- NumPy 배열을 PyTorch Tensor로 변환한다.
- 원본 NumPy 배열과 별도 Tensor를 만든다고 보면 된다.

> 기억할 점:  
> 1차원 Tensor의 shape은 `[5]`처럼 원소 개수만 표시된다.

## 6. 2차원 Tensor: Matrix

2차원 Tensor는 행과 열을 가진다.

강의 예시처럼 2행 3열 배열을 Tensor로 바꾼다.

In [ ]:
r2_np = np.array([
    [1, 5, 6],
    [4, 3, 2]
])

r2 = torch.tensor(r2_np).float()

print("NumPy array:")
print(r2_np)
print("NumPy shape:", r2_np.shape)

print("\nTensor:")
print(r2)
print("Tensor shape:", r2.shape)
print("Tensor dim:", r2.dim())
print("Tensor data:")
print(r2.data)

출력 해석:

```text
torch.Size([2, 3])
```

- 2행
- 3열

> 시험 포인트:  
> 2차원 Tensor는 행렬이고, 선형대수와 딥러닝 연산의 기본 형태다.

## 7. 3차원 Tensor 만들기

3차원 Tensor는 2차원 행렬이 여러 장 쌓인 구조로 볼 수 있다.

### 함수 사용법: `torch.randn()`

```python
torch.randn((3, 2, 2))
```

- 평균 0, 표준편차 1의 정규분포 난수를 만든다.
- 괄호 안에는 만들고 싶은 shape을 넣는다.

In [ ]:
torch.manual_seed(123)

r3 = torch.randn((3, 2, 2))

print("r3:")
print(r3)
print("shape:", r3.shape)
print("dim:", r3.dim())
print("numel:", r3.numel())

출력 해석:

```text
shape = [3, 2, 2]
```

- 2x2 행렬이 3개 있다고 보면 된다.
- `numel()`은 전체 원소 수를 의미한다.
- 여기서는 `3 × 2 × 2 = 12`개다.

### 함수 사용법: `numel()`

```python
tensor.numel()
```

- Tensor 안 전체 원소 개수를 반환한다.
- reshape/view를 할 때 원소 개수가 맞는지 확인할 때 유용하다.

## 8. 4차원 Tensor와 이미지 데이터

딥러닝에서 컬러 이미지는 보통 4차원 Tensor로 다룬다.

일반적인 이미지 Tensor 구조는 다음과 같다.

```text
(batch, channel, height, width)
```

- `batch`: 이미지 개수
- `channel`: 색상 채널, RGB면 3
- `height`: 세로
- `width`: 가로

강의 자료에서도 컬러 이미지를 다루기 위해 데이터 개수, 채널, 세로, 가로 4개 축이 필요하다고 설명한다.

In [ ]:
r4 = torch.ones((2, 3, 2, 2))

print("r4:")
print(r4)
print("shape:", r4.shape)
print("dim:", r4.dim())

출력 해석:

```text
shape = [2, 3, 2, 2]
```

- 이미지 2장
- RGB 채널 3개
- 세로 2
- 가로 2

> 기억할 점:  
> PyTorch 이미지 모델에서는 보통 `(N, C, H, W)` 순서를 사용한다.

## 9. Tensor 생성 함수 정리

PyTorch에는 Tensor를 만드는 여러 함수가 있다.

| 함수 | 사용법 | 의미 |
|---|---|---|
| `torch.tensor()` | `torch.tensor([1,2,3])` | 직접 값으로 Tensor 생성 |
| `torch.zeros()` | `torch.zeros(2, 3)` | 0으로 채운 Tensor |
| `torch.ones()` | `torch.ones(2, 3)` | 1로 채운 Tensor |
| `torch.full()` | `torch.full((2,3), 7)` | 특정 값으로 채운 Tensor |
| `torch.rand()` | `torch.rand(2, 3)` | 0~1 균등분포 난수 |
| `torch.randn()` | `torch.randn(2, 3)` | 정규분포 난수 |
| `torch.arange()` | `torch.arange(1, 7)` | 범위 Tensor |

In [ ]:
t_zeros = torch.zeros(2, 3)
t_ones = torch.ones(2, 3)
t_full = torch.full((2, 3), 7)
t_rand = torch.rand(2, 3)
t_randn = torch.randn(2, 3)
t_arange = torch.arange(1, 7)

print("zeros:")
print(t_zeros)

print("\nones:")
print(t_ones)

print("\nfull:")
print(t_full)

print("\nrand:")
print(t_rand)

print("\nrandn:")
print(t_randn)

print("\narange:")
print(t_arange)

> 실습 포인트:  
> 초기값을 만들 때는 `zeros`, `ones`, `full`을 쓰고,  
> 랜덤 초기화나 더미 데이터에는 `rand`, `randn`을 많이 쓴다.

## 10. Tensor 속성 확인하기

Tensor를 다룰 때는 항상 다음 정보를 확인하는 습관이 좋다.

```python
tensor.shape
tensor.size()
tensor.dim()
tensor.dtype
tensor.device
tensor.numel()
```

| 속성/함수 | 의미 |
|---|---|
| `shape` | Tensor 모양 |
| `size()` | Tensor 모양 |
| `dim()` | 차원 수 |
| `dtype` | 데이터 타입 |
| `device` | CPU/GPU 위치 |
| `numel()` | 전체 원소 개수 |

In [ ]:
x = torch.randn(3, 4, 5)

print("shape:", x.shape)
print("size:", x.size())
print("dim:", x.dim())
print("ndimension:", x.ndimension())
print("dtype:", x.dtype)
print("device:", x.device)
print("numel:", x.numel())

> 헷갈림 포인트:  
> `shape`과 `size()`는 거의 같은 정보를 준다.  
> `dim()`은 차원 개수만 알려준다.

## 11. dtype 변환: float와 long

딥러닝에서 데이터 타입은 중요하다.

- 입력값이나 weight는 보통 `float32`를 사용한다.
- 분류 문제의 정답 label은 보통 `long`, 즉 `int64`를 사용한다.

### 함수 사용법

```python
tensor.float()
tensor.long()
```

- `.float()`: `torch.float32`로 변환한다.
- `.long()`: `torch.int64`로 변환한다.

In [ ]:
print("원래 r1:")
print(r1)
print("dtype:", r1.dtype)

r5 = r1.long()

print("\nlong 변환 후 r5:")
print(r5)
print("dtype:", r5.dtype)

> 기억할 점:  
> 모델 입력은 float, 분류 label은 long이 자주 나온다.  
> dtype이 안 맞으면 PyTorch에서 에러가 날 수 있다.

## 12. NumPy 배열과 Tensor 변환

NumPy 배열과 Tensor는 서로 변환할 수 있다.

### NumPy → Tensor

```python
torch.tensor(np_array)
torch.from_numpy(np_array)
```

### Tensor → NumPy

```python
tensor.numpy()
```

또는 원본 강의 코드처럼:

```python
tensor.data.numpy()
```

를 볼 수 있다.

In [ ]:
np_data = np.array([[1, 2, 3], [4, 5, 6]])

tensor_from_np = torch.tensor(np_data)
tensor_from_numpy = torch.from_numpy(np_data)

print("np_data:")
print(np_data)

print("\ntorch.tensor:")
print(tensor_from_np)

print("\ntorch.from_numpy:")
print(tensor_from_numpy)

back_to_np = tensor_from_np.numpy()

print("\nTensor -> NumPy:")
print(back_to_np)
print(type(back_to_np))

> 주의:  
> `torch.from_numpy()`는 NumPy 배열과 메모리를 공유할 수 있다.  
> 값이 같이 바뀌는 상황이 생길 수 있으므로 원본 보존이 필요한 경우 조심해야 한다.

## 13. item() 함수

`item()`은 원소가 하나뿐인 Tensor에서 Python 숫자를 꺼내는 함수다.

### 함수 사용법

```python
tensor.item()
```

- 원소가 하나인 Tensor에만 사용할 수 있다.
- Loss 값을 기록하거나 출력할 때 자주 사용한다.

In [ ]:
print("r0:", r0)

item_val = r0.item()

print("item value:", item_val)
print("item type:", type(item_val))

t1 = torch.ones(1)

print("\n원소가 하나인 1차원 Tensor:")
print(t1)
print("item:", t1.item())

원소가 여러 개인 Tensor에는 `item()`을 사용할 수 없다.

아래 코드는 일부러 에러를 확인하는 예시다.

In [ ]:
try:
    print(r1.item())
except RuntimeError as e:
    print("item() 에러:")
    print(e)

> 필기 포인트:  
> `loss.item()`은 학습 기록을 남길 때 정말 자주 나온다.  
> Tensor 값 하나를 그냥 Python 숫자로 꺼내는 용도다.

## 14. Tensor 형태 변경: view()

`view()`는 Tensor의 shape을 바꾸는 함수다.

NumPy의 `reshape()`와 비슷하다.

### 함수 사용법

```python
tensor.view(new_shape)
```

- 전체 원소 개수는 유지되어야 한다.
- `-1`을 넣으면 해당 차원을 자동 계산한다.

In [ ]:
print("원본 r3 shape:", r3.shape)
print("원본 원소 개수:", r3.numel())

r6 = r3.view(3, 4)

print("\nr3.view(3, 4):")
print(r6)
print("shape:", r6.shape)

왜 가능한가?

```text
원본 r3 원소 개수 = 3 × 2 × 2 = 12
변경 후 3 × 4 = 12
```

원소 개수가 같으므로 바꿀 수 있다.

## 15. view(-1)과 자동 계산

`-1`은 PyTorch가 해당 차원의 크기를 자동 계산하라는 뜻이다.

In [ ]:
r6_auto = r3.view(3, -1)
r7 = r3.view(-1)

print("r3.view(3, -1):")
print(r6_auto)
print("shape:", r6_auto.shape)

print("\nr3.view(-1):")
print(r7)
print("shape:", r7.shape)

해석:

```text
r3.view(3, -1)
전체 12개
3 × ? = 12
? = 4
```

그래서 shape은 `[3, 4]`가 된다.

`view(-1)`은 전체 원소를 한 줄로 펼친다.

## 16. view()와 reshape() 차이

`view()`는 메모리가 연속적인 Tensor에서 잘 작동한다.  
전치한 Tensor처럼 메모리 배치가 연속적이지 않으면 에러가 날 수 있다.

이럴 때는 다음 중 하나를 사용한다.

```python
tensor.contiguous().view(...)
tensor.reshape(...)
```

- `contiguous()`: 메모리를 연속적으로 다시 배치한다.
- `reshape()`: 가능한 경우 view처럼, 필요하면 복사해서 처리한다.

In [ ]:
x = torch.randn(2, 3)
y = x.t()

print("x:")
print(x)
print("x shape:", x.shape)

print("\ny = x.t():")
print(y)
print("y shape:", y.shape)
print("y is contiguous?:", y.is_contiguous())

try:
    print(y.view(-1))
except RuntimeError as e:
    print("\nview 에러:")
    print(e)

print("\ncontiguous() 후 view:")
print(y.contiguous().view(-1))

print("\nreshape:")
print(y.reshape(-1))

> 헷갈림 포인트:  
> `view()`는 모양만 바꾸는 것 같지만, 내부 메모리 배치 조건이 있다.  
> 전치 후 에러가 나면 `contiguous()`나 `reshape()`를 떠올리면 된다.

## 17. squeeze와 unsqueeze

차원을 줄이거나 늘릴 때 자주 쓰는 함수다.

### 함수 사용법: `unsqueeze()`

```python
tensor.unsqueeze(dim)
```

- 지정한 위치에 크기 1인 차원을 추가한다.

### 함수 사용법: `squeeze()`

```python
tensor.squeeze()
tensor.squeeze(dim)
```

- 크기가 1인 차원을 제거한다.
- `dim`을 지정하면 해당 차원만 제거한다.

In [ ]:
exam = torch.rand(3, 1, 4, 1)

print("원본 shape:", exam.shape)

exam_s1 = exam.squeeze(dim=1)
print("squeeze(dim=1):", exam_s1.shape)

exam_s2 = exam_s1.squeeze(dim=-1)
print("squeeze(dim=-1):", exam_s2.shape)

exam_u = exam_s2.unsqueeze(dim=0)
print("unsqueeze(dim=0):", exam_u.shape)

> 기억할 점:  
> 모델 입력 shape을 맞출 때 `unsqueeze()`가 자주 나온다.  
> 예를 들어 `[N]`을 `[N, 1]`로 만들거나, 이미지에 channel 차원을 추가할 때 사용한다.

## 18. expand와 repeat

차원을 늘려서 같은 값을 여러 번 쓰고 싶을 때 사용한다.

### 함수 사용법: `expand()`

```python
tensor.expand(new_shape)
```

- 실제 데이터를 복사하지 않고 view처럼 확장해 보여준다.
- 크기 1인 차원만 확장할 수 있다.

### 함수 사용법: `repeat()`

```python
tensor.repeat(repeat_counts)
```

- 실제로 데이터를 반복해서 복사한다.

In [ ]:
origin_data = torch.randint(0, 4, (3, 2))

temp_data = origin_data.unsqueeze(0)

exp_res_data = temp_data.expand(4, 3, 2)
rep_res_data = temp_data.repeat(4, 1, 1)

print("origin_data:")
print(origin_data)
print("shape:", origin_data.shape)

print("\ntemp_data:")
print(temp_data)
print("shape:", temp_data.shape)

print("\nexpand 결과 shape:", exp_res_data.shape)
print(exp_res_data)

print("\nrepeat 결과 shape:", rep_res_data.shape)
print(rep_res_data)

차이 정리:

- `expand`: 메모리 절약에 유리하다.
- `repeat`: 실제로 복사하므로 메모리를 더 쓴다.

> 필기 포인트:  
> 단순히 같은 값을 broadcast처럼 쓰고 싶으면 expand를 먼저 생각한다.

## 19. Tensor 분할: chunk와 split

Tensor를 여러 조각으로 나눌 수 있다.

### 함수 사용법: `torch.chunk()`

```python
torch.chunk(tensor, chunks)
```

- Tensor를 지정한 개수의 덩어리로 나눈다.

### 함수 사용법: `torch.split()`

```python
torch.split(tensor, split_size)
```

- 각 조각의 크기를 기준으로 나눈다.

In [ ]:
tensor = torch.tensor([1, 2, 3, 4, 5, 6])

chunks = torch.chunk(tensor, 3)
splits = torch.split(tensor, 2)

print("원본:", tensor)

print("\nchunk(tensor, 3):")
for chunk in chunks:
    print(chunk)

print("\nsplit(tensor, 2):")
for split in splits:
    print(split)

차이:

```text
chunk: 몇 덩어리로 나눌지
split: 한 덩어리 크기를 얼마로 할지
```

## 20. Tensor 연결: torch.cat()

여러 Tensor를 하나로 붙일 때 `torch.cat()`을 사용한다.

### 함수 사용법

```python
torch.cat((a, b), dim=0)
torch.cat((a, b), dim=1)
```

- `dim=0`: 행 방향으로 붙인다.
- `dim=1`: 열 방향으로 붙인다.

In [ ]:
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[5, 6], [7, 8]])

cat_dim0 = torch.cat((a, b), dim=0)
cat_dim1 = torch.cat((a, b), dim=1)

print("a:")
print(a)

print("\nb:")
print(b)

print("\ncat dim=0:")
print(cat_dim0)
print("shape:", cat_dim0.shape)

print("\ncat dim=1:")
print(cat_dim1)
print("shape:", cat_dim1.shape)

> 헷갈림 포인트:  
> `dim=0`은 행 개수가 늘어나고, `dim=1`은 열 개수가 늘어난다.  
> NumPy의 `axis`와 비슷하게 이해하면 된다.

## 21. 기본 연산과 행렬 곱

Tensor도 NumPy처럼 element-wise 연산과 행렬 곱을 할 수 있다.

### 자주 쓰는 함수

```python
a + b
a - b
a * b
a / b
a ** 2
torch.dot(a, b)
torch.matmul(A, B)
```

- `*`: 같은 위치끼리 곱한다.
- `torch.dot`: 1차원 벡터 내적이다.
- `torch.matmul`: 행렬 곱이다.

In [ ]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

print("a + b:", a + b)
print("a - b:", a - b)
print("a * b:", a * b)
print("a / b:", a / b)
print("a ** 2:", a ** 2)
print("dot:", torch.dot(a, b))

A = torch.tensor([[1, 2, 3], [4, 5, 6]])
B = torch.tensor([[7, 8], [9, 10], [11, 12]])

C = torch.matmul(A, B)

print("\nmatmul 결과:")
print(C)
print("shape:", C.shape)

> 딥러닝 연결:  
> Linear Layer의 핵심도 결국 행렬 곱과 bias 더하기다.

## 22. 브로드캐스팅

브로드캐스팅은 shape이 다른 Tensor끼리 연산할 때 작은 Tensor를 자동으로 맞춰 계산하는 기능이다.

### 예시

```python
a.shape = (2, 2)
b.shape = (2,)
a + b
```

이 경우 `b`가 각 행에 자동으로 더해진다.

In [ ]:
a = torch.tensor([[1, 2], [3, 5]])
b = torch.tensor([1, 2])

c_add = a + b
c_mul = a * b

print("a shape:", a.shape)
print("b shape:", b.shape)

print("\na + b:")
print(c_add)

print("\na * b:")
print(c_mul)

> 필기 포인트:  
> 브로드캐스팅은 딥러닝에서 bias를 batch 전체에 더하는 구조와 연결된다.

## 23. 3차원 Tensor와 브로드캐스팅

3차원 Tensor와 2차원 Tensor도 브로드캐스팅될 수 있다.

In [ ]:
a = torch.randn(3, 4, 5)
b = torch.randn(4, 5)

c = a + b

print("a shape:", a.shape)
print("b shape:", b.shape)
print("c shape:", c.shape)

해석:

```text
a: (3, 4, 5)
b:    (4, 5)
```

PyTorch는 `b`를 `(1, 4, 5)`처럼 보고 자동 확장한다.

> 주의:  
> 뒤쪽 차원부터 비교해서 크기가 같거나 1이면 브로드캐스팅이 가능하다.

## 24. Tensor 슬라이싱

Tensor에서도 NumPy처럼 인덱싱과 슬라이싱을 한다.

### 사용법

```python
tensor[0, :]
tensor[:, 1]
tensor[1, 2]
tensor[0, :2]
```

In [ ]:
tensor = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

slice1 = tensor[0, :]
slice2 = tensor[:, 1]
slice3 = tensor[1, 2]
slice4 = tensor[0, :2]

print("tensor:")
print(tensor)

print("\n첫 번째 행:", slice1)
print("두 번째 열:", slice2)
print("두 번째 행 세 번째 열:", slice3)
print("첫 번째 행 앞 두 개:", slice4)

> 헷갈림 포인트:  
> PyTorch와 NumPy 모두 인덱스가 0부터 시작한다.  
> `tensor[1, 2]`는 두 번째 행, 세 번째 열이다.

## 25. 집계 함수: sum, max, min

Tensor도 합계, 최댓값, 최솟값을 계산할 수 있다.

### 함수 사용법

```python
torch.sum(tensor)
torch.sum(tensor, dim=0)
torch.max(tensor)
torch.min(tensor)
```

- `dim`: 계산할 방향을 지정한다.
- NumPy의 `axis`와 비슷한 개념이다.

In [ ]:
tensor = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

print("tensor:")
print(tensor)

print("\n전체 합:", torch.sum(tensor))
print("dim=0 합:", torch.sum(tensor, dim=0))
print("dim=1 합:", torch.sum(tensor, dim=1))

print("\n전체 max:", torch.max(tensor))
print("전체 min:", torch.min(tensor))

2차원 기준:

```text
dim=0: 세로 방향으로 계산해서 행 차원이 사라진다.
dim=1: 가로 방향으로 계산해서 열 차원이 사라진다.
```

> 시험 포인트:  
> PyTorch의 `dim`은 NumPy의 `axis`와 비슷하다.

## 26. torch.max로 최댓값과 인덱스 찾기

`torch.max()`는 딥러닝 분류에서 매우 자주 사용한다.

### 함수 사용법

```python
torch.max(tensor)
torch.max(tensor, dim)
```

- 인자 없이 쓰면 전체 최댓값을 반환한다.
- `dim`을 지정하면 각 방향별 최댓값과 인덱스를 반환한다.

In [ ]:
print("r2:")
print(r2)

print("\n전체 최댓값:")
print(r2.max())

max_dim0 = torch.max(r2, 0)
max_dim1 = torch.max(r2, 1)

print("\ndim=0 기준:")
print(max_dim0)

print("\ndim=1 기준:")
print(max_dim1)

print("\ndim=1 values:")
print(torch.max(r2, 1)[0])

print("\ndim=1 indices:")
print(torch.max(r2, 1)[1])

`torch.max(r2, 1)`의 반환값은 두 개다.

```text
values: 최댓값
indices: 최댓값 위치
```

다중 분류에서는 다음 패턴을 자주 쓴다.

```python
pred = torch.max(output, 1)[1]
```

> 기억할 점:  
> 모델 출력에서 가장 큰 점수를 가진 class 번호를 고를 때 사용한다.

## 27. device 속성

Tensor는 CPU 또는 GPU 위에 있을 수 있다.

### 함수 사용법

```python
torch.device("cuda" if torch.cuda.is_available() else "cpu")
tensor.to(device)
```

- `torch.cuda.is_available()`: GPU 사용 가능 여부를 확인한다.
- `.to(device)`: Tensor를 해당 장치로 이동한다.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tensor_device = torch.rand(3, 3).to(device)

print("device:", device)
print("tensor device:", tensor_device.device)

> 실무 포인트:  
> 모델과 데이터가 서로 다른 device에 있으면 에러가 난다.  
> 예를 들어 모델은 GPU, 데이터는 CPU에 있으면 계산할 수 없다.

## 28. requires_grad 속성

`requires_grad`는 자동 미분 추적 여부를 나타낸다.

### 사용법

```python
x = torch.tensor(3.0, requires_grad=True)
```

- `True`: 이 Tensor와 관련된 연산을 추적한다.
- `False`: gradient를 계산하지 않는다.

In [ ]:
print("r1.requires_grad:", r1.requires_grad)
print("r1.device:", r1.device)

x = torch.tensor(3.0, requires_grad=True)

print("\nx:", x)
print("x.requires_grad:", x.requires_grad)

> 필기 포인트:  
> 학습할 파라미터나 미분을 구하고 싶은 값에는 `requires_grad=True`가 필요하다.

## 29. 자동 미분 흐름

PyTorch 자동 미분 과정은 다음 순서로 이해하면 된다.

```text
1. requires_grad=True 설정
2. Tensor 연산 수행
3. 계산 그래프 자동 생성
4. backward() 호출
5. .grad에서 gradient 확인
6. zero_()로 gradient 초기화
```

강의 자료에서 강조한 부분도 이 흐름이다.

In [ ]:
autograd_steps = [
    "requires_grad=True 설정",
    "Tensor 연산 수행",
    "계산 그래프 생성",
    "backward() 호출",
    ".grad로 gradient 확인",
    "zero_()로 gradient 초기화",
]

for i, step in enumerate(autograd_steps, 1):
    print(f"{i}. {step}")

## 30. 2차 함수 경사 계산 준비

이번에는 강의 예시처럼 다음 함수를 미분한다.

```text
y = 2x² + 2
```

미분 결과는 다음과 같다.

```text
y' = 4x
```

먼저 x값을 NumPy 배열로 만든 뒤 Tensor로 바꾼다.

In [ ]:
x_np = np.arange(-2, 2.1, 0.25)

x = torch.tensor(
    x_np,
    requires_grad=True,
    dtype=torch.float32
)

print("x_np:")
print(x_np)

print("\nx Tensor:")
print(x)
print("requires_grad:", x.requires_grad)

### 함수 사용법: `torch.tensor(..., requires_grad=True, dtype=...)`

```python
torch.tensor(x_np, requires_grad=True, dtype=torch.float32)
```

- `x_np`: Tensor로 바꿀 NumPy 배열이다.
- `requires_grad=True`: 이 Tensor의 연산을 추적한다.
- `dtype=torch.float32`: 데이터 타입을 float32로 지정한다.

## 31. 2차 함수 계산과 grad_fn

Tensor로 연산하면 결과 Tensor에 `grad_fn`이 붙는다.

`grad_fn`은 이 Tensor가 어떤 연산으로 만들어졌는지에 대한 정보다.

In [ ]:
y = 2 * x**2 + 2

print("y:")
print(y)

print("\ny.grad_fn:")
print(y.grad_fn)

`grad_fn`이 보인다는 것은 PyTorch가 계산 그래프를 만들고 있다는 뜻이다.

> 헷갈림 포인트:  
> `requires_grad=True`인 Tensor로 만든 결과는 보통 `grad_fn` 정보를 가진다.

## 32. 2차 함수 그래프 그리기

`x.data`와 `y.data`를 사용해 그래프를 그린다.

In [ ]:
plt.plot(x.data, y.data, label="y = 2x^2 + 2")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Quadratic Function")
plt.legend()
plt.show()

그래프 해석:

- 아래로 볼록한 U자 형태다.
- x=0에서 가장 낮다.
- 양쪽으로 갈수록 값이 커진다.

> 딥러닝 연결:  
> Loss surface도 낮은 지점을 찾아 내려가는 문제로 생각할 수 있다.

## 33. backward()를 위해 스칼라로 만들기

PyTorch에서 `backward()`를 바로 호출하려면 결과가 스칼라여야 한다.

현재 `y`는 여러 값을 가진 벡터다.  
그래서 `sum()`으로 하나의 값 `z`를 만든다.

### 함수 사용법

```python
z = y.sum()
z.backward()
```

- `sum()`: 여러 값을 하나의 스칼라로 합친다.
- `backward()`: 계산 그래프를 거꾸로 따라가며 gradient를 계산한다.

In [ ]:
z = y.sum()

print("z:", z)
print("z shape:", z.shape)
print("z.grad_fn:", z.grad_fn)

> 기억할 점:  
> Loss도 보통 최종적으로는 스칼라 하나다.  
> 그래서 `loss.backward()`가 가능한 것이다.

## 34. backward()로 gradient 계산하기

이제 `z.backward()`를 호출하면 x에 대한 gradient가 계산된다.

In [ ]:
z.backward()

print("x.grad:")
print(x.grad)

결과 해석:

```text
y = 2x² + 2
y' = 4x
```

따라서 `x.grad`는 각 x값에 4를 곱한 값과 같다.

예:

```text
x = -2 → grad = -8
x =  2 → grad =  8
```

## 35. 함수와 gradient 그래프 비교

원래 함수와 gradient를 같이 그린다.

In [ ]:
plt.plot(x.data, y.data, label="y = 2x^2 + 2")
plt.plot(x.data, x.grad.data, label="gradient = 4x")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Function and Gradient")
plt.legend()
plt.show()

그래프 해석:

- 원래 함수는 U자 곡선이다.
- gradient는 직선이다.
- x가 음수면 gradient도 음수다.
- x가 양수면 gradient도 양수다.
- x=0 근처에서 gradient는 0이다.

> 시험 포인트:  
> gradient는 현재 위치에서 어느 방향으로 값이 증가하는지 알려준다.  
> Loss를 줄이려면 보통 gradient의 반대 방향으로 파라미터를 수정한다.

## 36. Gradient 누적 문제

PyTorch는 `backward()`를 호출할 때 gradient를 덮어쓰지 않고 누적한다.

그래서 같은 Tensor로 다시 backward를 하면 gradient가 더해진다.

In [ ]:
y_again = 2 * x**2 + 2
z_again = y_again.sum()

z_again.backward()

print("두 번째 backward 후 x.grad:")
print(x.grad)

결과가 두 배처럼 커진 이유는 gradient가 누적되었기 때문이다.

> 헷갈림 포인트:  
> PyTorch는 gradient를 자동으로 0으로 초기화하지 않는다.  
> 그래서 학습 루프에서 `optimizer.zero_grad()`가 꼭 필요하다.

## 37. zero_()로 gradient 초기화하기

새로운 gradient를 계산하기 전에는 기존 gradient를 지워야 한다.

### 함수 사용법

```python
x.grad.zero_()
```

- `x.grad` 값을 0으로 직접 바꾼다.
- `_`가 붙은 함수는 보통 원본을 직접 수정하는 in-place 함수다.

In [ ]:
x.grad.zero_()

print("초기화 후 x.grad:")
print(x.grad)

> 시험 포인트:  
> `backward()` 후에는 `.grad`에 값이 저장된다.  
> 다음 계산 전에는 `zero_()` 또는 `optimizer.zero_grad()`로 초기화해야 한다.

## 38. Sigmoid 함수 경사 계산

이번에는 PyTorch 내장 Sigmoid 함수의 gradient를 계산한다.

### 함수 사용법: `torch.nn.Sigmoid()`

```python
sigmoid = torch.nn.Sigmoid()
y = sigmoid(x)
```

- Sigmoid 함수를 Layer 객체처럼 만든다.
- 입력 Tensor를 0과 1 사이 값으로 변환한다.

In [ ]:
sigmoid = torch.nn.Sigmoid()

y_sigmoid = sigmoid(x)

print("y_sigmoid:")
print(y_sigmoid)

print("\ny_sigmoid grad_fn:")
print(y_sigmoid.grad_fn)

Sigmoid는 이진 분류에서 자주 나온다.

```text
작은 값 → 0에 가까움
큰 값 → 1에 가까움
0 근처 → 0.5 근처
```

## 39. Sigmoid 그래프 그리기

Sigmoid 함수의 모양을 확인한다.

In [ ]:
plt.plot(x.data, y_sigmoid.data, label="sigmoid(x)")
plt.xlabel("x")
plt.ylabel("sigmoid(x)")
plt.title("Sigmoid Function")
plt.legend()
plt.show()

그래프 해석:

- S자 모양이다.
- x가 작으면 0에 가까워진다.
- x가 크면 1에 가까워진다.
- x=0 근처에서 가장 빠르게 변한다.

## 40. Sigmoid gradient 계산

이번에도 `sum()`으로 스칼라를 만든 뒤 `backward()`를 호출한다.

In [ ]:
z_sigmoid = y_sigmoid.sum()

z_sigmoid.backward()

print("Sigmoid gradient:")
print(x.grad)

Sigmoid의 gradient는 가운데에서 크고 양 끝에서 작다.

> 딥러닝 연결:  
> Sigmoid가 양 끝으로 가면 gradient가 작아지기 때문에, 깊은 신경망에서는 기울기 소실 문제가 생길 수 있다.

## 41. Sigmoid와 gradient 그래프 비교

Sigmoid 값과 gradient를 같이 그린다.

In [ ]:
plt.plot(x.data, y_sigmoid.data, label="sigmoid(x)")
plt.plot(x.data, x.grad.data, label="gradient")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Sigmoid and Gradient")
plt.legend()
plt.show()

그래프 해석:

- Sigmoid는 S자 곡선이다.
- Gradient는 x=0 근처에서 가장 크다.
- 양 끝으로 갈수록 gradient가 작아진다.

> 필기 포인트:  
> 그래프를 같이 보면 “함수값”과 “변화율”이 다르다는 것을 눈으로 볼 수 있다.

## 42. 직접 Sigmoid 함수 구현하기

내장 함수가 아니라 직접 공식으로 Sigmoid를 만들 수도 있다.

공식:

```text
sigmoid(x) = 1 / (1 + exp(-x))
```

### 함수 사용법: `torch.exp()`

```python
torch.exp(x)
```

- e의 x제곱을 계산한다.
- Tensor 전체에 element-wise로 적용된다.

In [ ]:
x.grad.zero_()

def custom_sigmoid(x):
    return 1 / (1 + torch.exp(-x))

y_custom = custom_sigmoid(x)

z_custom = y_custom.sum()
z_custom.backward()

print("custom sigmoid:")
print(y_custom)

print("\ncustom sigmoid gradient:")
print(x.grad)

> 기억할 점:  
> 직접 만든 함수라도 Tensor 연산으로 구성되어 있으면 PyTorch가 자동 미분할 수 있다.

## 43. custom sigmoid 그래프

직접 만든 Sigmoid와 gradient를 확인한다.

In [ ]:
plt.plot(x.data, y_custom.data, label="custom sigmoid")
plt.plot(x.data, x.grad.data, label="gradient")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Custom Sigmoid and Gradient")
plt.legend()
plt.show()

내장 Sigmoid와 직접 구현한 Sigmoid의 모양은 같다.

중요한 것은 PyTorch가 Tensor 연산을 추적하고, `backward()`로 gradient를 계산해준다는 점이다.

## 44. 기울기, 미분, 경사 차이

강의 자료에서는 기울기, 미분, 경사를 구분했다.

| 용어 | 의미 |
|---|---|
| 기울기 Slope | 직선에서 x가 1 증가할 때 y가 얼마나 변하는지 |
| 미분 Derivative | 곡선의 특정 한 점에서 접선의 기울기 |
| 경사 Gradient | 여러 변수에 대한 미분값을 모은 벡터 |

딥러닝에서는 파라미터가 보통 여러 개다.

예:

```text
Loss = f(W, B)
```

이럴 때 W에 대한 미분, B에 대한 미분을 각각 구하고,  
이들을 모아놓은 것이 gradient다.

In [ ]:
summary = {
    "Slope": "직선의 기울기",
    "Derivative": "곡선 한 점에서의 순간 변화율",
    "Gradient": "여러 변수에 대한 미분값을 모은 벡터"
}

for key, value in summary.items():
    print(f"{key}: {value}")

> 시험 포인트:  
> 머신러닝에서 Loss를 줄이는 방향을 찾기 위해 gradient가 필요하다.

## 45. Define-by-run 방식

PyTorch는 Define-by-run, 즉 동적 계산 그래프 방식을 사용한다.

```text
코드가 실행되는 순간 계산 그래프가 만들어진다.
```

반대로 예전 TensorFlow 1.x 방식은 Define-and-run, 즉 그래프를 먼저 정의하고 나중에 실행하는 방식이었다.

PyTorch 방식의 장점:

- Python 코드 흐름과 비슷해서 직관적이다.
- 디버깅이 쉽다.
- 조건문, 반복문과 자연스럽게 섞어 쓸 수 있다.

In [ ]:
def dynamic_graph_example(x):
    if x.item() > 0:
        y = x ** 2
    else:
        y = -x
    return y

x_pos = torch.tensor(2.0, requires_grad=True)
x_neg = torch.tensor(-2.0, requires_grad=True)

y_pos = dynamic_graph_example(x_pos)
y_neg = dynamic_graph_example(x_neg)

y_pos.backward()
y_neg.backward()

print("x_pos grad:", x_pos.grad)
print("x_neg grad:", x_neg.grad)

이 예시는 입력값에 따라 다른 연산 그래프가 만들어질 수 있음을 보여준다.

> 필기 포인트:  
> PyTorch는 실행하면서 그래프가 만들어지기 때문에 코드가 직관적이다.

## 46. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `torch` | PyTorch 라이브러리 | `torch.tensor()`, `torch.randn()` |
| `Tensor` | PyTorch 데이터 컨테이너 | 숫자 배열 + 자동 미분 + GPU 지원 |
| `ndarray` | NumPy 배열 | Tensor와 비슷한 배열 구조 |
| `r0` | rank 0 Tensor | 0차원 스칼라 |
| `r1` | rank 1 Tensor | 1차원 벡터 |
| `r2` | rank 2 Tensor | 2차원 행렬 |
| `r3` | rank 3 Tensor | 3차원 Tensor |
| `shape` | Tensor 모양 | `tensor.shape` |
| `size()` | Tensor 모양 | `tensor.size()` |
| `dim()` | 차원 수 | `tensor.dim()` |
| `dtype` | 데이터 타입 | `tensor.dtype` |
| `device` | 저장 장치 | CPU 또는 GPU |
| `numel()` | 전체 원소 수 | `tensor.numel()` |
| `view()` | shape 변경 | `tensor.view(3, -1)` |
| `reshape()` | shape 변경 | `tensor.reshape(-1)` |
| `contiguous()` | 메모리 연속화 | 전치 후 view 전에 사용 |
| `squeeze()` | 크기 1 차원 제거 | `tensor.squeeze()` |
| `unsqueeze()` | 크기 1 차원 추가 | `tensor.unsqueeze(0)` |
| `expand()` | 가상 확장 | 메모리 복사 적음 |
| `repeat()` | 실제 반복 복사 | 메모리 사용 증가 |
| `cat()` | Tensor 연결 | `torch.cat((a,b), dim=0)` |
| `max()` | 최댓값 | `torch.max(tensor, dim=1)` |
| `indices` | 최댓값 위치 | `torch.max(...)[1]` |
| `requires_grad` | 자동 미분 추적 | `requires_grad=True` |
| `grad` | gradient 저장 위치 | `x.grad` |
| `backward()` | 역전파 | `loss.backward()` |
| `zero_()` | 원본을 0으로 초기화 | `x.grad.zero_()` |
| `Sigmoid` | 0~1 변환 함수 | `torch.nn.Sigmoid()` |
| `Autograd` | 자동 미분 기능 | PyTorch가 gradient 계산 |

## 47. 시험용 요약

```text
Tensor = NumPy 배열과 비슷하지만, 자동 미분과 GPU 연산을 지원하는 PyTorch의 핵심 데이터 구조
```

꼭 기억할 것:

- Tensor는 PyTorch의 기본 숫자 컨테이너다.
- 0차원 Tensor는 스칼라다.
- 1차원 Tensor는 벡터다.
- 2차원 Tensor는 행렬이다.
- 이미지 데이터는 보통 4차원 Tensor로 다룬다.
- PyTorch 이미지 Tensor 기본 형태는 `(batch, channel, height, width)`다.
- `torch.tensor()`는 값을 Tensor로 만든다.
- `.float()`는 float32 타입으로 바꾼다.
- `.long()`은 int64 타입으로 바꾼다.
- `shape`, `size()`, `dim()`, `dtype`, `device`를 확인하는 습관이 중요하다.
- `view()`는 Tensor 모양을 바꾼다.
- `view(-1)`은 1차원으로 펼친다.
- 전치 후 `view()`가 안 되면 `contiguous()`나 `reshape()`를 사용한다.
- `unsqueeze()`는 차원을 추가한다.
- `squeeze()`는 크기 1인 차원을 제거한다.
- `torch.max(output, 1)[1]`은 분류 예측 class를 구할 때 자주 쓴다.
- `requires_grad=True`는 자동 미분 추적을 시작한다.
- Tensor 연산을 하면 계산 그래프가 만들어진다.
- `backward()`는 gradient를 계산한다.
- `.grad`에는 계산된 gradient가 저장된다.
- gradient는 누적되므로 `zero_()`로 초기화해야 한다.
- `y = 2x² + 2`의 미분은 `4x`다.
- Sigmoid는 0과 1 사이 값을 출력한다.
- Sigmoid의 gradient는 가운데에서 크고 양 끝에서 작다.
- PyTorch는 Define-by-run 방식이라 코드 실행 중 계산 그래프를 만든다.